In [ ]:
# Upload the dataset
from google.colab import files

uploaded = files.upload()


In [ ]:
# Import the required libraries
!pip install lazypredict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_auc_score, recall_score
from tabulate import tabulate
import time
from sklearn.metrics import roc_auc_score, recall_score, f1_score
from lazypredict.Supervised import LazyClassifier
import os
from google.colab import drive
from sklearn.metrics import accuracy_score, balanced_accuracy_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, ExtraTreesClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis, LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifierCV, RidgeClassifier, Perceptron, PassiveAggressiveClassifier
from sklearn.neighbors import NearestCentroid, KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyClassifier


In [ ]:
# Load the CSV file into a pandas DataFrame
file_name = 'df_train.csv'
df = pd.read_csv(file_name)

df.info()


# Lazy Classifier


In [ ]:
# The 21 classifiers to be screened

# 1. Mount Google Drive so the results are stored permanently
drive.mount('/content/drive')

# Use a dedicated file name to avoid mixing the results with earlier runs
csv_path = '/content/drive/MyDrive/hasil_evaluasi_model_fulldata.csv'

# === [1] Use the full dataset (no sub-sampling) ===
print("Class distribution of the full dataset:")
print(df['Target'].value_counts())

# === [2] Separate the features from the target ===
X = df.drop(columns=['Target'])
y = df['Target']

# === [3] Train-test split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123, stratify=y
)

classes = np.unique(y_train)
y_test_bin = label_binarize(y_test, classes=classes)


In [ ]:
# === [4] The 21 classifiers (n_jobs=-1 speeds up the full-data run) ===
models_dict = {
    'RandomForestClassifier': RandomForestClassifier(random_state=123, n_jobs=-1),
    'XGBClassifier': XGBClassifier(random_state=123, verbosity=0, n_jobs=-1),
    'LGBMClassifier': LGBMClassifier(random_state=123, verbose=-1, n_jobs=-1),
    'BaggingClassifier': BaggingClassifier(random_state=123, n_jobs=-1),
    'DecisionTreeClassifier': DecisionTreeClassifier(random_state=123),
    'ExtraTreesClassifier': ExtraTreesClassifier(random_state=123, n_jobs=-1),
    'GaussianNB': GaussianNB(),
    'QuadraticDiscriminantAnalysis': QuadraticDiscriminantAnalysis(),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=123, n_jobs=-1),
    'LinearDiscriminantAnalysis': LinearDiscriminantAnalysis(),
    'NearestCentroid': NearestCentroid(),
    'LinearSVC': LinearSVC(max_iter=2000, random_state=123),
    'CalibratedClassifierCV': CalibratedClassifierCV(n_jobs=-1),
    'SGDClassifier': SGDClassifier(random_state=123, n_jobs=-1),
    'RidgeClassifierCV': RidgeClassifierCV(),
    'RidgeClassifier': RidgeClassifier(random_state=123),
    'Perceptron': Perceptron(random_state=123, n_jobs=-1),
    'PassiveAggressiveClassifier': PassiveAggressiveClassifier(random_state=123, n_jobs=-1),
    'KNeighborsClassifier': KNeighborsClassifier(n_jobs=-1),
    'AdaBoostClassifier': AdaBoostClassifier(random_state=123),
    'DummyClassifier': DummyClassifier(strategy='prior')
}

# === [5] Skip the models already evaluated in a previous run ===
completed_models = set()
if os.path.exists(csv_path):
    df_existing = pd.read_csv(csv_path)
    completed_models = set(df_existing['Model'].tolist())
    print(f"\nFound {len(completed_models)} models already evaluated in a previous run.")

# === [6] Evaluate the models one by one ===
for model_name, model_instance in models_dict.items():
    if model_name in completed_models:
        print(f"Skipping {model_name} (already stored)")
        continue

    print(f"Running {model_name}...")
    start_time = time.time()

    try:
        model_instance.fit(X_train, y_train)
        y_pred = model_instance.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        bal_acc = balanced_accuracy_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred, average='macro')
        f1 = f1_score(y_test, y_pred, average='macro')

        # ROC-AUC
        y_proba = None
        if hasattr(model_instance, "predict_proba"):
            y_proba = model_instance.predict_proba(X_test)
        elif hasattr(model_instance, "decision_function"):
            decision = model_instance.decision_function(X_test)
            if decision.ndim == 1:
                decision = np.vstack([-decision, decision]).T
            exp = np.exp(decision - np.max(decision, axis=1, keepdims=True))
            y_proba = exp / np.sum(exp, axis=1, keepdims=True)

        if y_proba is not None and y_proba.shape[1] == len(classes):
            auc = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')
        else:
            auc = None

        runtime = round(time.time() - start_time, 2)

        result_row = {
            'Model': model_name,
            'Accuracy': round(acc, 4),
            'Balanced Accuracy': round(bal_acc, 4),
            'ROC AUC Macro': round(auc, 4) if auc is not None else None,
            'Recall (Macro)': round(recall, 4),
            'F1-score (Macro)': round(f1, 4),
            'Time Taken (s)': runtime
        }

    except Exception as e:
        print(f"Failed to train {model_name}: {e}")
        result_row = {
            'Model': model_name,
            'Accuracy': None,
            'Balanced Accuracy': None,
            'ROC AUC Macro': None,
            'Recall (Macro)': None,
            'F1-score (Macro)': None,
            'Time Taken (s)': None
        }

    # Append the result of each model to the CSV file on Google Drive
    df_row = pd.DataFrame([result_row])
    file_exists = os.path.exists(csv_path)
    df_row.to_csv(csv_path, mode='a', header=not file_exists, index=False)

# === [7] Display the final results ===
df_final = pd.read_csv(csv_path)
display(df_final.sort_values(by='F1-score (Macro)', ascending=False))


In [ ]:
# Read the results from the evaluation DataFrame
# Alternatively, load them from the exported CSV file

# 1. Filter and sort the results by accuracy
plot_data = df_final.dropna(subset=['Accuracy']).sort_values('Accuracy', ascending=False)

# 2. Model names, taken either from the "Model" column or from the index
model_names = plot_data['Model'] if 'Model' in plot_data.columns else plot_data.index

# 3. Horizontal bar chart
plt.figure(figsize=(10, 8))
bars = plt.barh(model_names, plot_data['Accuracy'], color='skyblue')

plt.xlabel('Accuracy Score')
plt.ylabel('Model')
plt.title('Accuracy comparison across models')
plt.gca().invert_yaxis()  # Place the best model at the top

# Extra space on the right so the labels are not clipped
plt.xlim(0, 1.15)

# 4. Annotate each bar with its accuracy
for bar in bars:
    width = bar.get_width()
    plt.text(
        width + 0.01,                        # Slightly to the right of the bar
        bar.get_y() + bar.get_height() / 2,  # Vertically centred on the bar
        f"{width*100:.2f}%",                 # Accuracy as a percentage
        va='center', ha='left', fontsize=9, color='black'
    )

plt.tight_layout()
plt.show()
